<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/Makro_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
# Install necessary packages for Selenium in Colab
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium bs4 pandas
!pip install playwright beautifulsoup4 polars
!playwright install chromium
!playwright install-deps

# 1. Clear out the broken Ubuntu packages
!apt-get remove -y chromium-browser chromium-chromedriver

# 2. Download and install the official Google Chrome stable version
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

# 3. Install Python dependencies including the auto-installer
!pip install selenium bs4 pandas chromedriver-autoinstaller
!pip install polars xlsxwriter fastexcel
!pip install playwright beautifulsoup4 polars
!playwright install
!pip install itables
!playwright install-deps
!pip install curl_cffi

%%capture
# 1. Install all dependencies including pandas
!pip install xlsxwriter scrapling patchright msgspec browserforge nest_asyncio polars -q
!patchright install chromium
!patchright install-deps

### Import Lib

In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-09


In [3]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import re

async def scrape_makro_spa_clicker(start_url: str):
    extracted_data = []
    seen_names = set() # Track names so we don't scrape duplicates if the page is slow

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )

        page = await context.new_page()
        print("Walking through Makro's front door...")

        # Load the base URL ONCE
        await page.goto(start_url, wait_until="networkidle")
        await asyncio.sleep(6) # Wait for initial hydration

        assume_MAX_page = 16

        for page_num in range(1, assume_MAX_page + 1):
            print(f"Scraping page {page_num}...")

            # Scroll to the bottom to force lazy-loaded images and prices to render
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(2)

            # Take a snapshot of the DOM
            html_content = await page.content()
            soup = BeautifulSoup(html_content, "html.parser")

            # Makro wraps product cards in <a> tags linking to /p/ (product pages)
            product_cards = soup.find_all("a", href=lambda href: href and "/p/" in href)

            new_items_found = 0

            for card in product_cards:
                texts = list(card.stripped_strings)
                if len(texts) < 3:
                    continue

                # 1. Hunt for the Product Name
                name = None
                for text in texts:
                    if len(text) > 10 and "฿" not in text and "points" not in text.lower() and "Today" not in text:
                        name = text
                        break

                if not name or name in seen_names:
                    continue

                # 2. Hunt for the Prices (Hyper-Resilient Float Extractor)
                prices = []
                for text in texts:
                    if "buy" in text.lower() or "get" in text.lower() or "point" in text.lower():
                        continue

                    # Strip out currency symbols and commas
                    clean_text = text.replace("฿", "").replace(",", "").strip()

                    try:
                        val = float(clean_text)
                        # Safeguard: Ensure it's an actual price, not a quantity like "3 options"
                        if 5 < val < 10000:
                            prices.append(val)
                    except ValueError:
                        pass

                if not prices:
                    continue

                # 3. The E-Commerce Rule: Smallest is promo, Largest is original
                promo_price = min(prices)
                original_price = max(prices)

                # 4. Hunt for the Discount Condition (e.g., "2 - 2 units")
                condition = None

                # Primary strategy: Target the robust data-test-id you identified
                condition_tag = card.find(lambda tag: tag.has_attr("data-test-id") and "_lbl_buy_more" in tag["data-test-id"])

                if condition_tag:
                    condition = condition_tag.get_text(strip=True)
                else:
                    # Fallback strategy: Read through strings to catch anomalies like "3+ units"
                    for text in texts:
                        if "unit" in text.lower() and any(char.isdigit() for char in text):
                            condition = text
                            break

                # 5. Append everything to the dataset
                extracted_data.append({
                    "name": name,
                    "promotion_price": promo_price,
                    "original_price": original_price,
                    "condition": condition # Pushed the new column here
                })

                seen_names.add(name)
                new_items_found += 1

            print(f"  -> Extracted {new_items_found} new products.")

            # 6. THE SPA CLICKER
            if page_num < assume_MAX_page:
                try:
                    # Find the pagination "Next" button and click it
                    next_button = page.locator("text=Next").first

                    if await next_button.is_visible():
                        await next_button.click()
                        print("  -> Clicked 'Next'. Waiting for SPA to load...")
                        await asyncio.sleep(4) # Give React time to swap the products
                    else:
                        print("  -> 'Next' button not visible. Reached the end of the catalog!")
                        break
                except Exception as e:
                    print(f"  -> Pagination stopped: {e}")
                    break

        await browser.close()

    # --- POLARS CLEANUP ---
    df = pl.DataFrame(extracted_data)

    if not df.is_empty():
        df = df.unique(subset=["name"], maintain_order=True)

        # Nullify fake promotions
        df = df.with_columns(
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )

        # Calculate the Discount Percentage
        df = df.with_columns(
            pl.when(pl.col("original_price").is_not_null())
            .then(((pl.col("original_price") - pl.col("promotion_price")) / pl.col("original_price") * 100).round(1))
            .otherwise(None)
            .alias("discount_pct")
        ).sort("name")

    return df

In [4]:
# @title RUN create dataframe
# --- RUN IT ---
# Notice we DO NOT add "&page={}" to the URL anymore. We let the clicker do the work!
url_detergent = "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20DETERGENT%20Fresh%20Scent/17983?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwREVURVJHRU5UJTIwRnJlc2glMjBTY2VudCUyMiUyQyUyMmNhcm91c2VsVGl0bGUlMjIlM0ElMjJIb21lJTIwQ2FyZSUyMCU3QyUyMERFVEVSR0VOVCUyMEZyZXNoJTIwU2NlbnQlMjIlN0Q"

url_fresh_soft = "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Fresh%20and%20Soft/17985?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTdE"

df_makro_detergent = await scrape_makro_spa_clicker(start_url=url_detergent)

print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_detergent)} unique products")
print("=" * 60)
print(df_makro_detergent)

df_makro_freshsoft = await scrape_makro_spa_clicker(start_url=url_fresh_soft)
print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_freshsoft)} unique products")
print("=" * 60)
print(df_makro_freshsoft)

Walking through Makro's front door...
Scraping page 1...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 2...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 3...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 4...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 5...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 6...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 7...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 8...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 9...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 10...
  -> Extracted 20 new products.
  -> Clicked 'Next'

In [5]:
df_makro = pl.concat([df_makro_detergent, df_makro_freshsoft])
## Transform Makro
if "discount_pct" in df_makro.columns:
    df_makro = df_makro.drop("discount_pct")

In [6]:
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_makro = re_evaluate_price(df_makro)

In [7]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [8]:
df_trans_makro = parse_product_names(df_prep_makro, "Makro")

In [9]:
df_trans_makro

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""ALL Washing Powder Regular 3 k…",99.0,143.0,null,"""2026-04-09""","""ALL""",3,"""KG""",null,"""Makro"""
"""ARO Detergent for Washing Mach…",null,314.0,null,"""2026-04-09""","""ARO""",1,"""KG""","""8+1""","""Makro"""
"""ARO Detergent for Washing Mach…",null,419.0,null,"""2026-04-09""","""ARO""",1,"""KG""","""8+1""","""Makro"""
"""ARO Liquid Detergent Machine 3…",169.0,187.0,null,"""2026-04-09""","""ARO""",5,"""L""",null,"""Makro"""
"""ARO Liquid Fabric Washer 3.5 l""",169.0,182.0,null,"""2026-04-09""","""ARO""",5,"""L""",null,"""Makro"""
…,…,…,…,…,…,…,…,…,…
"""Hygiene Expert Care Forever Bl…",36.0,39.0,"""3 - 5 units""","""2026-04-09""","""Hygiene""",110,"""ML""","""X 3""","""Makro"""
"""Out of stock""",88.0,98.0,null,"""2026-04-09""","""Out""",null,null,null,"""Makro"""
"""PRO Fabric Softener Garden Swe…",25.0,27.0,"""2+ units""","""2026-04-09""","""PRO""",450,"""ML""","""X 3""","""Makro"""


In [10]:
df_trans_makro.write_excel(f"makro_{today_date}.xlsx")